## Evolutionary-track fitting with NUTS

Same observations as `fit_isochrone.ipynb`, using the existing local track NPZ.
The grid covers initial mass 0.8–1.5 solar masses, initial FeH −0.25–0.25, and EEP 202–550.
Run from within this repository, with a fresh kernel for the parallel-chain configuration.

In [ ]:
from pathlib import Path
import numpyro

num_chains = 4
numpyro.set_host_device_count(num_chains)  # before JAX initializes devices

import jax
jax.config.update("jax_enable_x64", True)

from jaxstar.mistfit import MistModel, MistFit

In [ ]:
repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/jaxstar").is_dir()
)
grid_path = repo_root / "benchmark-results/mist-track-grid-extended/grid.npz"
model = MistModel.from_tracks(grid_path)

In [ ]:
observations = {
    "teff": (5719, 110.),
    "feh": (-0.12, 0.1),
    "kmag": (8.843, 0.014),
    "parallax": (7.27355841, 3.008e-2),
}
mf = MistFit(model=model, observations=observations)

### NUTS

Mass, initial FeH and EEP bounds default to the grid limits. Age is an output;
`linear_age=True` uses the centered `dt/dEEP` field for the age prior.
EEP uses a fixed interval; the default soft upper age bound is 13.8 Gyr,
with `age_bound_width=0.1` Gyr. Set `"age_max": None` in `model_kwargs`
to disable it. `age_min` optionally adds a lower bound in Gyr.
Each bound has a log penalty of −15 at the boundary, approaching −30 outside;
it is a soft weight and also applies with `nodata=True`.


In [ ]:
mcmc = mf.run_nuts(
    jax.random.PRNGKey(0),
    num_chains=num_chains,
    #model_kwargs={"linear_age": True, "nodata": True},
    model_kwargs={"linear_age": True, "age_max": 13.8},
    output_keys=("kmag_abs",),
)

In [ ]:
mcmc.print_summary()

In [ ]:
idata = mf.to_inference_data(mcmc)
# Choose a new directory for each run, or pass overwrite=True explicitly.
result_dir = mf.save_result(idata, repo_root / "examples/mistfit/results/track_nuts", overwrite=True)

In [ ]:
outkeys = [*observations, "feh_init", "mass", "radius", "age", "eep"]
figures = mf.plot_diagnostics(idata, keys=outkeys, output_dir=result_dir, overwrite=True)

In [ ]:
_ = mf.plot_posterior_tracks(idata, y='kmag', eep_range=(250, 450))

In [ ]:
_ = mf.plot_corner(idata, keys=['mass', 'feh_init', 'age'])

## Nested sampling

In [ ]:
mn_dir = "results/track_multinest"
mn = mf.run_multinest(outputfiles_basename=mn_dir + "/run1_", n_live_points=100)

In [ ]:
mn_idata = mf.to_inference_data(mn)
mf.save_result(mn_idata, mn_dir, overwrite=True)
mn_figures = mf.plot_diagnostics(mn_idata, output_dir=mn_dir,
                                keys=("mass", "radius", "age", "feh_init"),
                                overwrite=True)

In [ ]:
from jaxstar.mistfit import get_run_metadata
get_run_metadata(mn_idata)["evidence"]